In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'torch', 'torchvision']
imports = {'pillow': 'PIL'}
pinned = {}
fallbacks = {'torch': 'torch==2.11.0', 'torchvision': 'torchvision==0.26.0'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "8a5255b0fce49fc730baae6ea6ecc601d45bb16b"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/8a5255b0fce49fc730baae6ea6ecc601d45bb16b/d2l"
for name in ('__init__.py', 'torch.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Multi-GPU from First Principles

Every training run so far has used one GPU, although this book's build
machine holds four. A run on one of them leaves three quarters of the machine's
arithmetic idle. The simplest way to use all four is to give every GPU a
copy of the model and a quarter of each minibatch. The copies must then
agree after every step, so the GPUs exchange their gradients, and the
exchange takes time that a single GPU never spends. When does a second GPU
shorten a step, and when does the exchange cost more than it saves?

On our machine the exchange is slow compared with the work. Each RTX 4090
performs about 83 trillion fp32 operations per second and reads its own
memory at 1 TB/s. Gradients travel between the cards over PCIe and through
host memory, because the cards cannot address each other directly
(that section). A PCIe 4.0 link carries a nominal
32 GB/s in each direction, about thirty times less than a GPU reads from
its own memory. A second GPU shortens training only if the computation it
takes over lasts longer than the exchange it adds.

This section builds the scheme, called *data parallelism*, by hand. We
split each minibatch, compute gradients on every device, sum them with a
collective operation called an *allreduce*, and apply the same update
everywhere. We then derive the allreduce algorithm that collective
libraries use. Finally, a cost model predicts the time of a step from
three quantities that we measure: the time a device needs for its share
of the minibatch, the fixed cost of one allreduce, and the bandwidth
between devices. Our test case is LeNet on Fashion-MNIST. Its steps are
so short that fixed costs dominate them, which makes it the hardest case
for data parallelism and the easiest one to take apart.

In [ ]:
import math
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

In [ ]:
import time

In [ ]:
torch.set_float32_matmul_precision('high')

## Three Ways to Split

A model and its minibatch can be divided among devices in three ways
(the figure).

![Three ways to split a model across two devices, with the cells of each
device in its own color: replicate the model and split the batch (data
parallel); split the layer stack (pipeline parallel); split every layer's
width (tensor parallel).](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-perf-splitting.svg)

Data parallelism gives every device a full copy of the model and a
different slice of the minibatch. Each device runs the whole network on
its slice, the devices sum their gradients, and every copy takes the same
optimizer step. It works for any model that fits on one GPU, and the
devices communicate once per step. *Pipeline parallelism* splits the stack of
layers: each device holds a few consecutive layers and passes its
activations to the next device. *Tensor parallelism* splits every layer:
each device holds a slice of each weight matrix and computes a slice of
each layer's output, so the devices communicate several times per layer.
The last two let a model that is too large for one GPU run at all.
AlexNet used such a split in 2012: each of its layers was divided between
two GPUs, because the network did not fit into one card's 3 GB
[@Krizhevsky.Sutskever.Hinton.2012]. Today pipeline and tensor
parallelism matter at the scale of the models in the Language Models part,
and we leave them to it.

Data parallelism does not let us train a larger model. Every device holds
a full copy of the parameters, the gradients, and the optimizer state,
which take 16 bytes per parameter for fp32 training with Adam
(that section). A model that does not fit on one GPU
therefore does not fit on two. What data parallelism offers is speed: $k$
devices share the work of each minibatch.

## Data Parallelism by Hand

A data-parallel step on $k$ devices has four parts
(the figure). We split the minibatch into $k$ equal
shards. Each device computes the gradient of the loss on its shard,
starting from its own copy of the parameters. The devices then sum their
$k$ gradients, so that every device holds the total. Finally, every device
applies the same update. Since all copies start equal and receive the same
update, they stay equal without ever being compared.

![One data-parallel step on two GPUs. Split the minibatch, run identical
forward and backward passes on each replica, sum the gradients with an
allreduce, then apply the identical update
everywhere.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-perf-data-parallel.svg)

We build each part with explicit tensor operations, so that every byte that
moves between devices appears in the code.

### The Model and the Data

We use the LeNet of that section: two convolutional layers with
average pooling, followed by three fully connected layers. We make two of the
substitutions that the table lists for modern networks:
ReLU activations in place of sigmoids, and He initialization
(that section) in place of Xavier initialization.
Instead of a module, we write the network as a function of a list of
parameter tensors. The parameters, and later their gradients, are then
plain tensors that we can copy from device to device.

In [ ]:
def new_params():
    """LeNet's parameters on the CPU: He-initialized weights, zero biases."""
    params = []
    for shape in [(6, 1, 5, 5), (16, 6, 5, 5), (120, 400), (84, 120),
                  (10, 84)]:
        fan_in = math.prod(shape[1:])
        params += [torch.randn(shape) * math.sqrt(2 / fan_in),
                   torch.zeros(shape[0])]
    return params

def lenet(params, X):
    W1, b1, W2, b2, W3, b3, W4, b4, W5, b5 = params
    H = F.avg_pool2d(F.relu(F.conv2d(X, W1, b1, padding=2)), 2)
    H = F.avg_pool2d(F.relu(F.conv2d(H, W2, b2)), 2).flatten(1)
    H = F.relu(F.linear(H, W3, b3))
    H = F.relu(F.linear(H, W4, b4))
    return F.linear(H, W5, b5)

def to_device(split, device):
    """A whole Fashion-MNIST split on `device`: images in [0, 1], labels."""
    return split.data.to(device).unsqueeze(1) / 255, split.targets.to(device)

LeNet is small. Its 61,706 parameters in ten tensors take 247 KB in fp32.
The 60,000 training images take 188 MB as floats, little enough to keep a
copy on every GPU. A copy on every GPU takes the input pipeline out of the
measurements below: each device reads its share of a minibatch from
its own copy, and the only data that cross between devices are gradients.

In [ ]:
assert d2l.num_gpus() >= 2, 'This section needs two GPUs'
data = d2l.FashionMNIST()
devices = [d2l.try_gpu(i) for i in range(2)]
train_data = [to_device(data.train, d) for d in devices]  # A copy each
test_data = to_device(data.val, devices[0])
base = new_params()          # One initialization for every run below
print(f'{sum(p.numel() for p in base)} parameters in {len(base)} tensors; '
      f'images {tuple(train_data[0][0].shape)} on each of 2 devices')

### Broadcast and Allreduce

Two operations make the step parallel. *Broadcast* copies one device's
tensors to every other device; we need it once, to give every device the
same initial parameters. Allreduce takes one tensor per device, sums
them, and leaves the sum on every device; we need it once per step, for
the gradients. The simplest allreduce is a *star*: device 0 collects the
tensors of all other devices and adds them up, then sends the sum back.

In [ ]:
def broadcast(params, device):
    """A copy of every tensor in `params` on `device`, ready for autograd."""
    return [p.to(device, copy=True).requires_grad_() for p in params]

def allreduce(data):
    """Sum the tensors in `data`, one per device, onto every device."""
    for i in range(1, len(data)):    # Star: collect and add on device 0 ...
        data[0] += data[i].to(data[0].device)
    for i in range(1, len(data)):    # ... then copy the sum back
        data[i].copy_(data[0])

`.to(device)` copies a tensor from one device to another; the copy and the
addition on device 0 are the whole algorithm.

Two vectors, one on each GPU, show what an allreduce leaves behind:

In [ ]:
x = [torch.full((1, 2), i + 1.0, device=d) for i, d in enumerate(devices)]
print('before:', [x_j.tolist() for x_j in x])
allreduce(x)
print('after: ', [x_j.tolist() for x_j in x])

Each device now holds $1 + 2 = 3$ in every entry.

### The Data-Parallel Step

Each device computes the gradient of the mean loss over its own shard.
With $k$ shards of equal size, the average of the $k$ shard gradients is
the gradient of the mean loss over the whole minibatch. An allreduce
followed by a division by $k$ therefore gives every device the full-batch
gradient.
(The exercises ask what changes when the shards differ in size.)

LeNet's gradient consists of ten tensors, and we could allreduce them one
at a time. Every allreduce has a fixed cost, however few bytes it moves,
so we instead concatenate each device's gradients into one flat vector and
make a single allreduce per step. that section measures
how much this saves. The training step computes every device's gradient on
its shard, sums the gradients with one allreduce, and applies the update.
Its argument `communicate=False` skips the allreduce, so that every device
updates its own copy from its own shard alone. The result is not a
training step, but it lets us time the work that the devices do without
communicating.

In [ ]:
def shard_grads(params, data, idx):
    """Gradient of the mean loss on each device's share of the minibatch
    `idx` (example indices), as one flat vector per device."""
    grads = []
    for ps, (X, y), i in zip(params, data, idx.chunk(len(params))):
        i = i.to(X.device, non_blocking=True)  # Don't wait for the device
        F.cross_entropy(lenet(ps, X[i]), y[i]).backward()
        grads.append(torch.cat([p.grad.flatten() for p in ps]))
        for p in ps:
            p.grad = None              # The next backward starts from zero
    return grads

def train_step(params, data, idx, lr, communicate=True):
    """One data-parallel SGD step on the minibatch `idx`; without
    communication, every device steps on its own shard's gradient."""
    grads = shard_grads(params, data, idx)
    if communicate:
        allreduce(grads)               # One message per device and step
    with torch.no_grad():
        for ps, g in zip(params, grads):
            for p, g_p in zip(ps, g.split([p.numel() for p in ps])):
                p -= lr / len(params) * g_p.view_as(p)  # Mean of k grads

`shard_grads` computes the gradient on every device in turn and flattens
it. It then clears `p.grad`, because `backward` adds to the gradients that
are already stored. The step runs in a single Python process, which drives
all devices in turn: it issues the forward and backward pass for device 0,
then for device 1, then the copies of the allreduce. The kernels of
different devices can still run at the same time, because each GPU works
through its own queue while Python fills the next one.

### A One-Step Check

A data-parallel step should update the parameters as a step on one device
with the same minibatch does. We check this once, before
reading any timing. The check is cheap, and it catches errors that an
accuracy curve hides. A step that sums the shard gradients instead of
averaging them trains with a learning rate $k$ times too large, and it
may still reach a plausible accuracy. From one random initialization, we take
one step on one GPU and one step on two GPUs, on the same 256 examples.

The check runs in true fp32. Elsewhere in this section the GPU's tensor
cores multiply fp32 operands in tf32, with a 10-bit mantissa
(that section). A minibatch of 256 and one of 128 can select
different convolution algorithms, and their tf32 rounding errors would
hide the small difference that we look for.

In [ ]:
idx = torch.randperm(len(data.train))[:256]
updates = []
torch.set_float32_matmul_precision('highest')  # True fp32 in matmuls ...
with torch.backends.cudnn.flags(enabled=True, allow_tf32=False):  # ... convs
    for k in (1, 2):
        params = [broadcast(base, d) for d in devices[:k]]
        train_step(params, train_data[:k], idx, lr=0.2)
        updates.append([p.detach().cpu() - b
                        for p, b in zip(params[0], base)])
torch.set_float32_matmul_precision('high')
largest = max(u.abs().max() for u in updates[0])
diff = max((u1 - u2).abs().max() for u1, u2 in zip(*updates))
print(f'largest update {largest:.1e}, largest difference {diff:.1e}')

The two updates differ by less than a hundred-thousandth of the largest
update. That is fp32 rounding, because the two steps add the same numbers
in a different order. The parallel step computes what the single-device step
computes, so any difference in speed that we measure below comes from how
the work is spread over the devices.

## Ring Allreduce

A star allreduce routes everything through device 0. For a gradient of
$N$ bytes on $k$ devices, device 0 receives $(k-1)N$ bytes and sends
$(k-1)N$ bytes, while the links of the other devices carry $N$ bytes each
way and then sit idle. The time of the star therefore grows linearly
in $k$. *Ring allreduce* [@Patarasuk.Yuan.2009] spreads the traffic
evenly over all devices.

Arrange the $k$ devices in a ring in which each sends only to its
successor, and split every device's gradient vector into $k$ chunks of
$N/k$ bytes. The allreduce runs in two phases
(the figure). In the first phase, called
*reduce-scatter*, each device sends one chunk to its successor and adds
the chunk it receives from its predecessor to its own copy of that chunk.
After $k-1$ such steps, every device owns one chunk that holds the sum
over all devices, a different chunk on each device. In the second phase,
called *all-gather*, the finished chunks travel around the ring for
another $k-1$ steps until every device has all of them. At every step
every device sends one chunk and receives one chunk, so each device sends
$(k-1)N/k$ bytes per phase, and the two phases together send

$$
\frac{2(k-1)}{k}\, N \quad\textrm{bytes per device.}
$$

The traffic per device thus does not grow with $k$: it is $N$ bytes at
$k = 2$ and approaches $2N$ from below as $k$ grows. For a ResNet-18,
with 45 MB of fp32 gradients, on four GPUs each device sends 67 MB, where
the hub of a star would send 134 MB and receive as much. The two phases
are also useful on their own. Reduce-scatter leaves each device with the
sum of one part of the gradient, and all-gather assembles a vector from
parts held on different devices. The sharded training of the next section
uses them separately.

![Ring allreduce on four GPUs, computed step by step. Reduce-scatter
accumulates one chunk per device around the ring; all-gather then
distributes the finished chunks. Each cell counts how many of the four
contributions a chunk holds, so a chunk holding 4 is complete. An
outlined cell is the chunk that a GPU received in that step, and the ring
in the last panel shows which GPU sends to which.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-perf-ring-allreduce.svg)

The chunk bookkeeping is easy to get wrong, so we check it in a simulation.
Each NumPy vector stands for the gradient on one device, and one step of
the ring copies every message before any device adds or stores what it
receives, since all devices send at once. In reduce-scatter, device $j$
sends chunk $j - s$ in step $s$, so device $j$ ends with the finished
chunk $j + 1$, which it sends first in all-gather.

In [ ]:
def ring_allreduce(xs):
    """Ring allreduce of the vectors xs, one per device; returns every
    device's result and the bytes that each device sent."""
    k = len(xs)
    chunks = [np.array_split(x.copy(), k) for x in xs]  # chunks[j][c]
    sent = 0
    for s in range(k - 1):  # Reduce-scatter: add the received chunk
        msgs = [chunks[j][(j - s) % k].copy() for j in range(k)]
        for j, msg in enumerate(msgs):
            chunks[(j + 1) % k][(j - s) % k] += msg
        sent += msgs[0].nbytes
    for s in range(k - 1):  # All-gather: replace by the received chunk
        msgs = [chunks[j][(j + 1 - s) % k].copy() for j in range(k)]
        for j, msg in enumerate(msgs):
            chunks[(j + 1) % k][(j + 1 - s) % k] = msg
        sent += msgs[0].nbytes
    return [np.concatenate(c) for c in chunks], sent

Four devices with vectors of 1,024 floats, $N = 4{,}096$ bytes each,
should all end with the sum and each send
$2 \cdot 3/4 \cdot N = 6{,}144$ bytes:

In [ ]:
rng = np.random.default_rng()
xs = [rng.standard_normal(1024, dtype=np.float32) for _ in range(4)]
results, sent = ring_allreduce(xs)
print(f'every device holds the sum: '
      f'{all(np.allclose(r, sum(xs), atol=1e-5) for r in results)}; '
      f'sent per device: {sent} bytes, '
      f'2(k - 1)N/k = {2 * 3 * xs[0].nbytes // 4} bytes')

Every device ends with the sum, and each sends exactly the bytes of
the equation.

The byte count does not determine the time on its own. The ring takes
$2(k-1)$ steps one after another, and each step includes the fixed time of
a transfer, so the fixed cost of a ring allreduce grows linearly in $k$
while its bytes do not. For small messages, where the fixed cost dominates, collective
libraries switch to algorithms with fewer sequential steps, such as
trees, whose depth grows only like $\log_2 k$; an exercise derives where
the crossover lies. On our machine, the ring also has fewer independent links
to spread its traffic over than its picture suggests. Without peer-to-peer
access, a transfer from one GPU to its neighbor goes up one PCIe link
into host memory and down another (that section),
so all hops share the host's memory system and two PCIe host bridges.
the equation says how many bytes each device must move. How
fast they move is a property of the machine that we have to measure.

## Communication Cost and Scaling

We can now ask the question of the opening: does a second GPU shorten a
step? The answer comes from comparing the computation that moves to the
new device with the communication that it adds.

### A Cost Model for One Step

On one device, a step on a minibatch of $B$ examples takes time
$t_{\textrm{compute}}(B)$ for the device's local work: the forward and
backward pass and the update. On $k$ devices, each device does this work
for $B/k$ examples, and the step also waits for the allreduce of the $N$
gradient bytes that every device holds. A transfer of $n$ bytes takes time
$\alpha + n/\beta$, a fixed cost $\alpha$ plus the bytes divided by a
bandwidth $\beta$ [@Hockney.1994].

We apply this model to the allreduce as a whole and write its time as $\alpha + 2N/\beta$, which
defines $\beta$ by measurement: $2N/\beta$ is the time of an allreduce so
large that its fixed cost does not matter. On two devices, every allreduce
makes each device send $N$ bytes and receive $N$, so $\beta$ counts the
bytes that leave and enter a device together. For our star, whose two
copies run one after the other, $\beta$ is the bandwidth of one copy. The
step time on $k$ devices is then approximately

$$
t_{\textrm{step}}(k) \approx t_{\textrm{compute}}(B/k) + m\,\alpha +
\frac{2N}{\beta},
$$

where $m$ is the number of allreduce calls per step, each with the fixed
cost $\alpha$. At $k = 1$ both communication terms vanish. Only the first
term shrinks as devices are added; the other two stay. A second device
therefore shortens the step only if
$t_{\textrm{compute}}(B) - t_{\textrm{compute}}(B/2)$, the computation it
takes over, exceeds $m\alpha + 2N/\beta$, the communication it adds. The
three quantities $t_{\textrm{compute}}$, $\alpha$, and $\beta$
can each be measured on their own, and we do so for LeNet.

### Latency and Bandwidth of the Allreduce

An allreduce of a single number moves almost no bytes, so its time is
$\alpha$. An allreduce of 64M floats, 256 MB per device, takes so long
that $\alpha$ hardly matters, and $\beta = 2N/(t - \alpha)$ follows from
its time $t$. We measure both on two GPUs. A training loop does not wait
for the devices after each allreduce; the next kernels queue behind
it. We therefore time many calls in a row and wait once at the end, since
a wait after every call would add its own delay to $\alpha$.

In [ ]:
def allreduce_time(n, devices, calls):
    """Seconds per allreduce of n floats per device."""
    x = [torch.zeros(n, device=d) for d in devices]
    return d2l.Benchmark(lambda: allreduce(x), warmup=10 * calls,
                         repeats=3 * calls, devices=devices).time

alpha = allreduce_time(1, devices, calls=100)
n = 2**26                                # 64M floats ...
N = 4 * n                                # ... 256 MB
beta = 2 * N / (allreduce_time(n, devices, calls=3) - alpha)
print(f'alpha = {1e6 * alpha:.0f} us, beta = {beta / 1e9:.1f} GB/s')

`d2l.Benchmark` waits for every device in `devices` before it starts and
before it stops the clock. It warms up with ten times as many calls as it
times, because the first few hundred allreduces of a run can be slower than
the rest, which would inflate $\alpha$. The star allreduce reaches about 20 GB/s.
On two devices it consists of two copies of $N$ bytes, one after the
other, so each copy runs at about 20 GB/s as well. The GPUs' copy engines
move the bytes over PCIe and through host memory, and this is about what
one PCIe 4.0 link carries in one direction after that detour. The fixed
cost $\alpha$ is that of two driver-managed copies.

At these bandwidths, moving LeNet's 247 KB of gradients takes
$2N/\beta$, at most about a tenth of a millisecond. Does it matter whether
they travel as one message or as ten? We allreduce LeNet's ten gradient
tensors first one at a time, then all at once as one concatenated vector.
The cell prints both times next to the predictions of
the equation:

In [ ]:
grads = [[torch.zeros_like(p, device=d) for p in base] for d in devices]
flat = [torch.cat([g.flatten() for g in G]) for G in grads]
def separately():
    for g in zip(*grads):              # One allreduce per tensor
        allreduce(list(g))
n_bytes, m = flat[0].nbytes, len(base)
for name, f, calls in (('separately', separately, m),
                       ('at once', lambda: allreduce(flat), 1)):
    t = d2l.Benchmark(f, warmup=1000, repeats=300, devices=devices).time
    print(f'{name:10s}: {1e6 * t:4.0f} us measured, '
          f'{1e6 * (calls * alpha + 2 * n_bytes / beta):4.0f} us predicted')

Ten calls take several times as long as one call that moves the same
bytes.

The two constants predict both times to within a few percent.

For a gradient this small, the number of messages decides the communication time, and the
bytes hardly matter. This is why collective libraries pack many small
gradient tensors into a few large buffers before they send them, as our
training step does by hand.

### LeNet on One and Two GPUs

A step on two GPUs spends $\alpha + 2N/\beta$ on one allreduce. What it
saves is the difference between $t_{\textrm{compute}}(256)$, the time of
one device's local work on a minibatch of 256 examples, and
$t_{\textrm{compute}}(128)$, the time of the same work on half of it. We
measure both on one GPU with `communicate=False`. A third measurement runs
the local work on two GPUs at once, 128 examples each, which is what the
data-parallel step does before its allreduce:

In [ ]:
params, t_local = [broadcast(base, d) for d in devices], {}
for k, b in ((1, 256), (1, 128), (2, 128)):
    t_local[k, b] = d2l.Benchmark(
        lambda: train_step(params[:k], train_data[:k], idx[:k * b], 0.2,
                           communicate=False),
        warmup=500, repeats=150, devices=devices[:k]).time
    print(f'{k} GPU(s), {b} examples each: {1e3 * t_local[k, b]:.2f} ms')

On one GPU, halving the minibatch shortens the local work by far less
than half. A LeNet layer applied to a few hundred small images is far too
little work to occupy an RTX 4090, so each kernel finishes before the next
one is ready to start. The step is close to the overhead-bound regime of
that section: its time is set mostly by the number of
kernels launched rather than by the arithmetic inside them, and halving the
minibatch leaves the number of kernels unchanged.

On two GPUs, the local work takes about twice as long as on one GPU with
the same 128 examples. One Python thread launches the kernels of both
devices, one device after the other, so twice as many launches wait for a
single interpreter, and each GPU idles while Python serves the other.
the equation assumes that the devices work independently; here
they share one launcher.

On two GPUs the local work thus takes longer than on one GPU with the
whole minibatch, the allreduce adds $\alpha + 2N/\beta$ on top, and
the equation predicts that a step on two GPUs takes longer than a
step on one. The predicted step time is the
one-GPU step, plus the difference between the local work on two GPUs and
on one, plus $\alpha + 2N/\beta$. To test the prediction, we train for
five epochs on one GPU and on two, from the same initialization `base`,
with minibatches of 256 examples. Both runs visit the training set in the
same five random orders, drawn once, so that they differ only in rounding.
The function `train` takes the list of orders, one per epoch. It returns
the average time per step over all epochs but the first, which includes
start-up costs, and then the test accuracy:

In [ ]:
def train(params, train_data, test_data, orders, batch_size=256, lr=0.2):
    """Data-parallel SGD on the devices that hold `params` and `train_data`,
    one epoch per example order; returns seconds per step, test accuracy."""
    devices = [X.device for X, _ in train_data]
    times = []
    for order in orders:
        t0 = time.perf_counter()
        for idx in order.split(batch_size):
            train_step(params, train_data, idx, lr)
        for d in devices:  # Wait until every device has finished
            torch.cuda.synchronize(d)
        times.append((time.perf_counter() - t0)
                     / math.ceil(len(order) / batch_size))
    X, y = test_data
    with torch.no_grad():
        acc = (lenet(params[0], X).argmax(1) == y).float().mean().item()
    return sum(times[1:]) / (len(times) - 1), acc

In [ ]:
orders, t_step = [torch.randperm(len(data.train)) for _ in range(5)], {}
for k in (1, 2):
    params = [broadcast(base, d) for d in devices[:k]]
    t_step[k], acc = train(params, train_data[:k], test_data, orders)
    print(f'{k} GPU(s): {1e3 * t_step[k]:.2f} ms per step, '
          f'test accuracy {acc:.2f}')
predicted = (t_step[1] + t_local[2, 128] - t_local[1, 256]
             + alpha + 2 * n_bytes / beta)
print(f'2 GPUs predicted: {1e3 * predicted:.2f} ms per step')

The step on two GPUs is slower than the step on one, as the separately
measured terms predict. For LeNet, a second GPU makes training slower.

The measured step usually lies within a few percent of the prediction.
Both are bound by the one Python thread that drives the two GPUs, so they
follow the speed of a single CPU core, and on a machine busy with other
work the two can differ by a fifth or more.

Both runs reach the same test accuracy to within a point or two. They
start from the same parameters and see the examples in the same order, so
they differ only in rounding, which a thousand steps of SGD amplify. Two runs on one GPU differ about as much, because a
GPU does not always add the same numbers in the same order.

LeNet is too small for data parallelism. A second device shortens a step
when the computation on each shard is long compared with
$m\alpha + 2N/\beta$, and that requires two things. The per-device batch
must be large enough to keep each GPU busy, so that
$t_{\textrm{compute}}(B/k)$ is close to $t_{\textrm{compute}}(B)/k$. And the
model must perform many operations per byte of gradient, so that the
computation it moves outweighs the exchange it adds. LeNet performs 0.4
million multiply-adds per image, under two per byte of its gradient. A
ResNet-18 on Fashion-MNIST images enlarged to 64×64 pixels performs 2.2
billion multiply-adds per image with 45 MB of gradients, about fifty per
byte.

### Strong and Weak Scaling

Every reported speedup holds something fixed, and two conventions are
common. the equation holds the global minibatch $B$ fixed while $k$
grows: the same work, finished sooner if all goes well. This is *strong
scaling*, and it is what we measured for LeNet. The alternative holds the
per-device batch fixed, so that the global minibatch grows to
$kB_{\textrm{device}}$: more work per unit of time. This is *weak
scaling*, and its speedups are quoted as throughput, examples per second.
Weak scaling is the easier test, because $t_{\textrm{compute}}$ per device
does not shrink and only the communication has to stay small next to it.
A larger global minibatch, however, changes the optimization itself
(that section), so a gain in examples per second under weak
scaling is not a gain of the same size in the time it takes to reach a
given loss.

We can now predict from three measurements whether more GPUs shorten a
step, and we have seen why they do not for LeNet. Our hand-written
version also leaves room for improvement. It waits for the whole backward
pass before it communicates, although the gradients of the last layers
are ready long before those of the first.

It also drives every device from one Python thread, and its star routes
all traffic through device 0 instead of a ring.

that section replaces our version with the frameworks'
production tools and measures them on a model large enough to gain from
more GPUs.

## Summary

Data parallelism trains a copy of the model on every device, each on its
share of the minibatch. The copies stay equal because an allreduce sums
their gradients before every update, and a one-step check confirms that
the parallel step computes what one device computes. A ring
allreduce, a reduce-scatter followed by an all-gather, moves
$2(k-1)N/k$ bytes per device, at most $2N$ however many devices take
part. Whether another device shortens a step follows from
$t_{\textrm{step}}(k) \approx t_{\textrm{compute}}(B/k) + m\alpha + 2N/\beta$,
whose terms can each be measured. For LeNet, the fixed cost $\alpha$ of
an allreduce outweighs its bytes, which is why gradients travel as one
message. Halving LeNet's minibatch does not halve its computation either,
so a second GPU makes each step slower. Data parallelism shortens steps
for models whose per-device computation is long compared with the
exchange of their gradients. A reported speedup must say whether it holds
the global or the per-device batch fixed.

## Exercises

1. **Data versus pipeline parallelism.** Consider a stack of $L$ fully
   connected layers of width $d$, with fp32 weights and no biases, trained
   on two GPUs with minibatch size $B$. Data parallelism allreduces the
   gradients of all weights. Two-stage pipeline
   parallelism (the figure) places the first $L/2$ layers on one GPU and
   the rest on the other, so the forward pass sends the $B \times d$
   activations at the cut to the second GPU and the backward pass returns
   their gradient.
    1. Count the bytes that cross between the two GPUs per step in each
       scheme, and evaluate both for $L = 32$, $d = 4096$, and $B = 256$.
       For which relation between $B$, $L$, and $d$ does data parallelism
       move fewer bytes?
    1. In the pipeline, each GPU idles while the other computes. To
       shorten the idle time, split the minibatch into $m$ micro-batches
       [@Huang.Cheng.Bapna.ea.2019]. In the forward phase, the first
       GPU hands each micro-batch to the second as soon as it is done, so
       that the second GPU works on micro-batch $j$ while the first works
       on micro-batch $j+1$. The backward phase starts after the last
       forward pass and runs the same schedule from the second GPU to the
       first. Assume that every micro-batch takes the same time on each
       stage. Derive the fraction of the step during which each GPU idles,
       and evaluate it for $m = 1$ and $m = 8$.

1. **Gradient normalization.** Two recipes compute a data-parallel
   gradient from $k$ shards of a minibatch of $B$ examples. In recipe A,
   each device sums its per-example losses, the devices sum their
   gradients, and the update divides by $B$. In recipe B, which our
   training step uses, each device averages its per-example losses, and
   the devices average their gradients.
    1. Show that both recipes yield the gradient of the full-batch mean
       loss when every shard holds $B/k$ examples.
    1. A language model averages its loss over tokens. Two shards with the
       same number of sequences hold 3000 and 1000 tokens. Find the weight
       that recipe B gives a token on each shard, relative to the weight
       the token has in the full-batch token mean.
    1. Modify recipe B so that it is exact for any shard sizes, and
       identify the extra quantity that the devices must communicate
       before the update.

1. **Strong-scaling speedup.** Model the compute time of a shard of $s$
   examples as $t_{\textrm{compute}}(s) = t_0 + c\,s$, where $t_0$ does not
   shrink with the shard. Let each device send $b$ bytes per second over
   its link, while it receives at the same rate. This per-direction rate
   $b$ is not the $\beta$ of the equation, which counts the bytes
   in both directions. By the equation, one step at a fixed
   global batch $B$ takes
   $$t_{\textrm{step}}(k) = t_0 + \frac{c\,B}{k} + \frac{2(k-1)}{k}\,\frac{N}{b},$$
   with no communication at $k = 1$. The strong-scaling speedup is
   $S(k) = t_{\textrm{step}}(1) / t_{\textrm{step}}(k)$.
    1. Show that $S(k) > 1$ holds for one $k > 1$ exactly when it holds
       for every $k > 1$, and find the condition. Find the limit of $S(k)$
       as $k \to \infty$.
    1. A model with 11.2 million fp32 parameters has $t_0 = 2$ ms and
       $c = 0.2$ ms per example, and it is trained with $B = 256$. Compute
       $S(2)$, $S(4)$, and the limit for $b = 20$ GB/s and for
       $b = 5$ GB/s. Find the rate $b$ below which no number of GPUs
       speeds up this model.
    1. Sending the gradients in bf16 instead of fp32 halves $N$. For
       $b = 5$ GB/s, recompute $S(4)$ and the limit of $S(k)$, and
       state which of the two gains more from the halving.

1. **Parameter servers and the ring.** Before collective libraries became
   standard, distributed training often summed gradients through
   *parameter servers* [@Li.Andersen.Park.ea.2014]: $s$ server
   machines each hold a slice of $N/s$ bytes of the parameters. In every
   step, each of $k$ workers pushes its gradient for each slice to that
   slice's server, the server adds up the $k$ contributions and updates
   its slice, and every worker pulls the updated slices back.
    1. Count the bytes that each worker and each server send and receive
       per step. Find the number of servers at which a server's link is no
       busier than a worker's.
    1. Now let the $k$ workers act as the servers, each holding $1/k$ of
       the parameters. Show that the traffic of each device equals
       the equation, and match the push and pull phases to the
       two phases of ring allreduce.

5. [code] **Ring allreduce by hand.** ● This exercise needs four GPUs.
    The star pattern routes all traffic through device 0, while the ring
    of the figure spreads it over all $k$ devices.
    1. For $k$ devices holding $N$ bytes each, count the bytes that the
       star and the ring move in total, and the bytes that the busiest
       device sends and receives. The star starts its broadcast only after
       its sum is complete. Predict the ratio of their times under two
       assumptions. In the first, each device has one link that carries
       $b$ bytes per second in each direction at the same time, and all
       links work concurrently. In the second, the machine runs one
       transfer at a time at $b$ bytes per second.
    1. Write `ring_allreduce(data)` with the interface of `allreduce`:
       split each tensor into $k$ chunks with `chunk`, and implement
       reduce-scatter and all-gather with explicit `.to()` copies. Check it
       against `allreduce` for $k \in \{2, 3, 4\}$, guarding with
       `d2l.num_gpus()`.
    1. Time both functions with `d2l.Benchmark` on 64M floats per device
       at $k = 2$ and $k = 4$, and decide which assumption of the first
       part describes your machine.

6. [code] **Gradients in bf16.** ● Sending gradients in bf16
   halves their bytes but rounds them. In a ring, each reduce-scatter hop
   adds the incoming partial sum to the local chunk, so a bf16 ring
   rounds the partial sum to bf16 once per hop.
    1. At the parameters returned by `new_params`, compute the gradients
       of `lenet` on $k$ disjoint Fashion-MNIST shards of 128 examples
       each, and flatten each shard's gradients into one vector. Add the
       vectors one after another, rounding to bf16 after every
       addition, and measure the relative error of the result against an
       fp32 sum for $k \in \{2, 8, 64\}$. Repeat with each vector rounded
       to bf16 once and the sum accumulated in fp32. Explain how each
       error depends on $k$.
    1. Compare the error at $k = 8$ with the relative difference between
       the gradients of two shards, and decide whether the rounding matters
       for SGD.

1. **Latency in the allreduce.** ● The byte count of
   the equation ignores the fixed cost of starting a transfer.
   Let one transfer of $n$ bytes over a link take $\alpha + n/b$, where
   $b$ is the rate of one link in one direction.
    1. Derive the time $t_{\textrm{ring}}$ of a ring allreduce of $N$ bytes
       on $k$ devices, and describe how each of its two terms behaves as
       $k$ grows.
    1. A binary-tree allreduce sums up the tree and broadcasts back down.
       It takes $2\log_2 k$ steps, each of which sends the full $N$ bytes,
       so
       $$t_{\textrm{tree}} = 2\log_2 k \left(\alpha + \frac{N}{b}\right).$$
       For $k = 8$, $\alpha = 10\ \mu\textrm{s}$, and
       $b = 20$ GB/s, find the message size below which the tree is
       faster than the ring.
    1. The allreduce of the equation takes $\alpha + 2N/\beta$.
       The gradient size $N_{1/2}$ at which it spends half of its time on
       the fixed cost is called its half-performance length. Express
       $N_{1/2}$ in $\alpha$ and $\beta$, and compute it from the values
       that you measured in that section. How many
       copies of LeNet's gradient would one message have to carry to reach
       it? Production data-parallel libraries pack gradients into buffers
       of tens of megabytes (25 MB by default in PyTorch's
       `DistributedDataParallel`); judge that size against your
       $N_{1/2}$.

1. [extended] **Where data parallelism helps.** ● LeNet is too small for a
   second GPU to help. Widen it, timing `train_step` with
   `communicate=False` on one and on two GPUs to measure
   $t_{\textrm{compute}}(b)$, and find out which terms of
   the equation the width changes.
    1. Generalize `new_params` and `lenet` so that the channel counts of
       both convolutions and the widths of the two hidden dense layers
       scale with a factor $w$. Compute the gradient bytes $N(w)$ and the
       forward-pass multiply-adds per example, and show how their ratio
       depends on $w$.
    1. For $w \in \{1, 2, 4, 8, 16\}$, measure the time per step of
       `train` at $k = 1$ and $k = 2$ with global batch 256, and plot the
       speedup against $w$.
    1. Overlay the speedup that the equation predicts from your
       measured $t_{\textrm{compute}}(128)$ and $t_{\textrm{compute}}(256)$
       and from the $\alpha$ and $\beta$ measured in the allreduce
       experiment of that section. Find the speedup that
       the prediction approaches as $w$ grows. Find the achieved compute rate
       in FLOP/s below which the second GPU helps at large $w$.
    1. Repeat the prediction and the measurement at $w = 16$ with global
       batch 1024, and explain why the batch size raises the limit of the
       previous part while the width does not.

[Discussions](https://d2l.discourse.group/t/36791)